In [9]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------


from selenium.webdriver.common.by import By
import pandas as pd
from time import sleep
import datetime
from pandas import ExcelWriter
import os
import undetected_chromedriver as uc

# Python 3.6++**

In [11]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'GT SIB' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.10.0")

now=datetime.datetime.now()

filename= '{} data {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

# scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename, engine='openpyxl')



tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)


Running GT SIB Web Scraping Tool v.1.10.0


In [12]:
#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty

    return sqldict


In [13]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------
sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}   


regdict={
'GT SIB 1': 'https://www.sib.gob.gt/ConsultaDinamica/?cons=251',# more one column, change a little logic
'GT SIB 2': 'https://www.sib.gob.gt/ConsultaDinamica/?cons=252',
'GT SIB 3': 'https://www.sib.gob.gt/ConsultaDinamica/?cons=253',
'GT SIB 4': 'https://www.sib.gob.gt/ConsultaDinamica/?cons=254',
'GT SIB 5': 'https://www.sib.gob.gt/ConsultaDinamica/?cons=255',
'GT SIB 6': 'https://www.sib.gob.gt/ConsultaDinamica/?cons=257', 
'GT SIB 7': 'https://www.sib.gob.gt/ConsultaDinamica/?cons=258',
'GT SIB 8': 'https://www.sib.gob.gt/ConsultaDinamica/?cons=259',
'GT SIB 9': 'https://www.sib.gob.gt/ConsultaDinamica/?cons=260',
'GT SIB 10': 'https://www.sib.gob.gt/ConsultaDinamica/?cons=452',

    }


Typology={

        regulatorName+' 1': 'INSTITUCIONES BANCARIAS',
        regulatorName+' 2': 'SOCIEDADES FINANCIERAS',
        regulatorName+' 3': 'COMPANIAS DE ALMACENADORAS',
        regulatorName+' 4': 'COMPANIAS DE SEGUROS',
        regulatorName+' 5': 'CASAS DE CAMBIO',
        regulatorName+' 6': 'CASAS DE BOLSA',
        regulatorName+' 7': 'TARJETAS DE CREDITO',
        regulatorName+' 8': 'OTRAS INSTITUCIONES',
        regulatorName+' 9': 'GRUPOS FINANCIEROS',
        regulatorName+' 10': 'INSTITUCIONES DE MICROFINANZAS',

        }
processdate = now.strftime('%Y-%m-%d')




In [16]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder
import ssl

ssl._create_default_https_context = ssl._create_unverified_context
chromeOptions = uc.ChromeOptions()

driver = uc.Chrome(options=chromeOptions)


URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: certificate chain too long (_ssl.c:1000)>

In [6]:
# ensure required engines are available for reading Excel files
# %pip install xlrd openpyxl --quiet
df_total = pd.DataFrame(sqldict)
for reg in regdict:
        
    sleep(10)
    driver.get(regdict[reg])
    print(f' --- Working {reg} ---')
    sleep(10)
    try:
        driver.find_element(By.TAG_NAME, "vaadin-button").click()
    except Exception as e:
        meg = driver.find_element(By.TAG_NAME, "title")
        # print(f"[ERROR] : {e},{meg.get_attribute("outerHTML")}")
        raise RuntimeError(
        f"click on <vaadin-button> failed; title markup: {meg.get_attribute('outerHTML')}"
    ) from e
    sleep(5)
    dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]

    if len(dl_files)>0 and not dl_files[0].endswith('.tmp') and not dl_files[0].endswith('.crdownload'):
        sleep(3)
        print('[INFO] -- Check the download file --')

    else:
        print('[INFO] -- Maybe the file link is error, Change to xlsx -- ')
        driver.get(regdict[reg][:]+'x')
        dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
    
    if not dl_files:
        print('[ERROR] -- No downloaded files found, skipping.')
        driver.quit()
        continue

    file_path = dl_files[0]
    # select engine based on file extension to avoid ValueError
    ext = os.path.splitext(file_path)[1].lower()

    # Detect if the downloaded .xls is actually an HTML page (some servers return HTML error pages)
    try:
        with open(file_path, 'rb') as fh:
            header = fh.read(512).lstrip().lower()
        is_html = header.startswith(b'<') and b'html' in header
    except Exception as e:
        print(f"[ERROR] Could not read downloaded file header: {e}")
        # cleanup and continue
        # driver.quit()
        # for rem in os.listdir(tempfolder):
        #     os.remove(os.path.join(tempfolder, rem))
        break

    # try:
    if is_html:
        # parse the HTML table instead of treating it as an Excel file
        sleep(3)
        dfs = pd.read_html(file_path)
        if len(dfs) == 0:
            raise ValueError("No tables found in HTML content.")
        df = dfs[0]
        print(dfs[1].shape)
        columns = dfs[1].iloc[:1].values.flatten().tolist()
        dfs[1].columns = columns
        dfs[1] = dfs[1].drop(index=0).reset_index(drop=True)
        df_list =  pd.DataFrame(sqldict)
        df_list['Name'] = dfs[1]['Nombre de la Entidad'].tolist()
        df_list['Address_1'] = dfs[1]['Dirección de Oficina Central'].tolist()
        df_list['Phone'] = dfs[1]['Números teléfonicos'].tolist()
        df_list['Fax'] = dfs[1]['Números de Fax'].tolist()
        df_list['Website'] = dfs[1]['Página Web'].tolist()
        df_list['processdate'] = processdate
        df_list['ListName'] = Typology[reg] 
        df_list['RegCtry'] = reg.split(' ')[0]
        df_list['RegCode'] = reg.split(' ')[1]
        df_list['ListCode'] = reg.split(' ')[-1]
        df_list['RegulationType'] = 'Regulated'
        df_total = pd.concat([df_total, df_list], axis=0)
    else:
        if ext == '.xls':
            excel_file = pd.ExcelFile(file_path, engine='xlrd')
        elif ext in ['.xlsx', '.xlsm', '.xltx', '.xltm']:
            excel_file = pd.ExcelFile(file_path, engine='openpyxl')
        else:
            # fallback to openpyxl; pandas will raise a clear error for unsupported formats
            excel_file = pd.ExcelFile(file_path, engine='openpyxl')

        # If the sheet name isn't exactly "Sheet1", try to read the first sheet
        sheet_name = "Sheet1"
        if sheet_name not in excel_file.sheet_names:
            sheet_name = excel_file.sheet_names[0]
        df = excel_file.parse(sheet_name)


    # except Exception as e:
    #     print(f"[ERROR] Failed to parse file {file_path}: {e}")

        # driver.quit()
        # for rem in os.listdir(tempfolder):
        #     try:
        #         os.remove(os.path.join(tempfolder, rem))
        #     except Exception:
        #         pass
        # continue

    sleep(3)

    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))


NameError: name 'driver' is not defined

In [7]:

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)
#df=pd.DataFrame(sqldict)

df_total.fillna('', inplace=True)
df_total.to_excel(writer, 'SQL Ready', index=False)

writer.save()
writer.close()
sleep(3)

driver.quit()

C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_45088\1212653093.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_total.fillna('', inplace=True)
C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_45088\1212653093.py:7: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df_total.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [8]:
df_total.to_excel(filename)

In [19]:
# check which is different

key_value_counts = {key: len(values) for key, values in sqldict.items()}

# Print the counts
for key, count in key_value_counts.items():
    print(f"Key '{key}' has {count} values.")

Key 'bvdid' has 0 values.
Key 'priority' has 0 values.
Key 'ListLabel' has 0 values.
Key 'Typology' has 0 values.
Key 'EntryType' has 0 values.
Key 'Name' has 0 values.
Key 'InternalID_1' has 0 values.
Key 'InternalID_1_type' has 0 values.
Key 'InternalID_2' has 0 values.
Key 'InternalID_2_type' has 0 values.
Key 'InternalID_3' has 0 values.
Key 'InternalID_3_type' has 0 values.
Key 'CoType' has 0 values.
Key 'License_Type' has 0 values.
Key 'Address_1' has 0 values.
Key 'Address_2' has 0 values.
Key 'City' has 0 values.
Key 'Zip' has 0 values.
Key 'Cntry' has 0 values.
Key 'Phone' has 0 values.
Key 'Fax' has 0 values.
Key 'Website' has 0 values.
Key 'Email' has 0 values.
Key 'RegulationType' has 0 values.
Key 'RegulationTypeCode' has 0 values.
Key 'RegulationDate' has 0 values.
Key 'CancellationDate' has 0 values.
Key 'RegCtry' has 0 values.
Key 'RegCode' has 0 values.
Key 'ListCode' has 0 values.
Key 'ListLanguage' has 0 values.
Key 'ListValidityDate' has 0 values.
Key 'ListName' has